# Exercise 2 — Fetch and plot ZTF flare data

Download the IRSA ZTF light curve for **ZTF19abanrhr** (candidate optical counterpart associated with AGN J124942.3+344929 / GW190521) and plot magnitude vs time by filter.

```bash
uv run jupyter notebook notebooks/exercises/02_fetch_plot_ztf.ipynb
```

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "fetch_data.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root (scripts/fetch_data.py)")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "scripts"))
print("Repo root:", ROOT)

## Fetch from IRSA

In [ ]:
from fetch_data import fetch_ztf_irsa

csv_path = fetch_ztf_irsa(object_name="ZTF19abanrhr")
print("CSV:", csv_path)

In [ ]:
df = pd.read_csv(csv_path, comment="#")
print("Columns:", list(df.columns)[:25])
print("Rows:", len(df))
df.head()

## Light curve by filter

IRSA ZTF tables typically include `mjd`, `mag`, `magerr`, and `filtercode` (`zg`, `zr`, `zi`).

In [ ]:
# Normalize common IRSA / ztfquery column names
cols = {c.lower(): c for c in df.columns}

def col(*names):
    for n in names:
        if n.lower() in cols:
            return cols[n.lower()]
    raise KeyError(names)

mjd = df[col("mjd", "obsmjd")].astype(float)
mag = df[col("mag", "magpsf")].astype(float)
magerr = df[col("magerr", "sigmagpsf", "magerr")].astype(float) if any(
    n.lower() in cols for n in ("magerr", "sigmagpsf")
) else None
filt = df[col("filtercode", "filter", "fid")].astype(str)

# Optional quality cut if catflags present
if "catflags" in cols:
    ok = df[cols["catflags"]].fillna(0).astype(int) == 0
    mjd, mag, filt = mjd[ok], mag[ok], filt[ok]
    if magerr is not None:
        magerr = magerr[ok]

GPS_GW190521 = 1242442967.4
# MJD ≈ GPS/86400 + 40587.0 (GPS to MJD conversion used by LIGO)
MJD_GW190521 = GPS_GW190521 / 86400.0 + 40587.0

fig, ax = plt.subplots(figsize=(8, 4.2))
colors = {"zg": "C2", "zr": "C3", "zi": "C4", "g": "C2", "r": "C3", "i": "C4"}
for code, grp_idx in filt.groupby(filt).groups.items():
    key = str(code).lower()
    c = colors.get(key, None)
    yerr = magerr.loc[grp_idx] if magerr is not None else None
    ax.errorbar(
        mjd.loc[grp_idx],
        mag.loc[grp_idx],
        yerr=yerr,
        fmt="o",
        ms=3,
        alpha=0.85,
        color=c,
        label=str(code),
        elinewidth=0.6,
    )

ax.axvline(MJD_GW190521, color="0.3", ls="--", lw=1, label="GW190521")
ax.invert_yaxis()
ax.set_xlabel("MJD")
ax.set_ylabel("magnitude")
ax.set_title("ZTF19abanrhr light curve (IRSA)")
ax.legend(loc="best", fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

print(f"GW190521 MJD ≈ {MJD_GW190521:.3f}")
print(f"Light-curve MJD range: {mjd.min():.3f} → {mjd.max():.3f}")

### Try next

- Zoom to ±200 days around the GW time.
- Plot flux instead of magnitude (approximate with \(F \propto 10^{-0.4 m}\)).
- Compare g and r variability amplitudes in a quiet window vs near the candidate flare.